# Settings

## Import packages

In [1]:
#!pip install --upgrade git+https://github.com/m-wrzr/populartimes
#!pip install python-dotenv

  Cloning https://github.com/m-wrzr/populartimes to /private/var/folders/l_/zlzp_vvs4j31hx3pg4j26tch0000gn/T/pip-req-build-e7rv819q
  Running command git clone --filter=blob:none --quiet https://github.com/m-wrzr/populartimes /private/var/folders/l_/zlzp_vvs4j31hx3pg4j26tch0000gn/T/pip-req-build-e7rv819q
  Resolved https://github.com/m-wrzr/populartimes to commit 989f66b63dd1b6a144c5ad4a7c3a200f7f8649ee
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached urllib3-1.26.20-py2.py3-none-any.whl.metadata (50 kB)
Using cached urllib3-1.26.20-py2.py3-none-any.whl (144 kB)
  Created wheel for populartimes: filename=populartimes-2.0-py3-none-any.whl size=13609 sha256=04ab29ebc88b049bc3c8de3851b659aa959315be38a5b0e97fb22576aefd4a1f
  Stored in directory: /private/var/folders/l_/zlzp_vvs4j31hx3pg4j26tch0000gn/T/pip-ephem-wheel-cache-sg014qkc/wheels/29/17/f2/

In [52]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import math
import itertools
import geopandas as gpd
import matplotlib.patches as mpatches
import requests
import time
import populartimes
import os

from scipy import stats
from scipy.stats import chi2_contingency, pointbiserialr
from geopy.geocoders import Nominatim
from shapely.geometry import Point
from matplotlib.patches import Circle
from dotenv import load_dotenv

pd.set_option('display.precision', 4)

In [54]:
# duplicate the example.env file, rename it as '.env' and put your key in the '.env' file
# DO NOT push the .env file
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

## Self-defined functions

### fetch_google_place_id(api_key, place_name)

In [ ]:
def fetch_google_place_id(api_key, place_name):
    search_url = "https://maps.googleapis.com/maps/api/place/findplacefromtext/json"
    search_params = {
        "input": place_name,
        "inputtype": "textquery",
        "fields": "place_id",
        "key": api_key
    }
    
    try:
        response = requests.get(search_url, params=search_params, timeout=5)
        response.raise_for_status()
        sres = response.json()
        
        candidates = sres.get("candidates", [])
        if not candidates:
            return False  # No result found
        return candidates[0].get("place_id")
    
    except Exception as e:
        return False  # Error occurred

### gen_place_info_popularity_df

In [103]:
def gen_place_info_popularity_df(response):
    """
    Converts a populartimes.get_id() response into two DataFrames:
    
    - place_info_df: Basic info about the place (ID, name, rating, location, etc.)
    - popularity_df: Hourly popularity data per weekday (hour_0 to hour_23)
    
    Parameters:
        response (dict): Result from populartimes.get_id()
    
    Returns:
        tuple: (place_info_df, popularity_df)
    """
    place_info_df = pd.DataFrame([{
        'place_id': response['id'],
        'name': response['name'],
        'address': response['address'],
        'types': ','.join(response['types']),
        'lat': response['coordinates']['lat'],
        'lon': response['coordinates']['lng'],
        'rating': response['rating'],
        'review_count': response['rating_n'],
        'phone': response.get('international_phone_number', ''),
        'time_spent_min': response.get('time_spent', [None])[0],
        'time_spent_max': response.get('time_spent', [None, None])[1]
    }])

    popularity_rows = []
    for day_data in response.get('populartimes', []):
        row = {'place_id': response['id'], 'day': day_data['name']}
        row.update({f'hour_{hour}': value for hour, value in enumerate(day_data['data'])})
        popularity_rows.append(row)

    popularity_df = pd.DataFrame(popularity_rows)
    
    return place_info_df, popularity_df

# Google API
- Apply for API Key:
  1. [Create a Google Cloud Project](https://console.cloud.google.com/projectcreate?utm_source=Docs_NewProject&utm_content=Docs_elevation-backend&hl=zh-tw&_gl=1*3gdz2p*_ga*MTY0MTk4NDM1My4xNzQ4OTQ2MzEz*_ga_NRWSTWS78N*czE3NDkwNTkyMDUkbzQkZzEkdDE3NDkwNTk5OTYkajQ5JGwwJGgw&inv=1&invt=AbzPGA)
  2. [Enable Places API + Get Your API Key](https://developers.google.com/maps/documentation/places/web-service/get-api-key?)
- [View SKU Usage and Cost Report](https://console.cloud.google.com/billing/01A4EF-B501A6-60B1D3/reports;grouping=GROUP_BY_SKU?inv=1&invt=AbzO6A)
- [Place Details (legacy) – fields and structure](https://developers.google.com/maps/documentation/places/web-service/legacy/details?hl=zh-tw#fields)
- [(❗️Important❗️) Google Maps Platform Pricing (official)](https://mapsplatform.google.com/pricing/)! 

In [7]:
restaurant_df = pd.read_csv('dataset/Manhattan_Restaurant_Latest_Inspection_Results_cleaned.csv')
# Use DBA + BUILDING + STREET to generate full name
restaurant_df['full_name'] = restaurant_df['DBA'] + ' ' + restaurant_df['BUILDING'] + ' ' + restaurant_df['STREET']
restaurant_df['INSPECTION DATE'] = pd.to_datetime(restaurant_df['INSPECTION DATE'])

# Remove records with invalid inspection date
restaurant_df = restaurant_df[restaurant_df['INSPECTION DATE'] != pd.to_datetime('1900-01-01')]
valid_restaurant_list = restaurant_df[['full_name']].drop_duplicates().reset_index(drop=True)

# Add columns for further use
for col in ["place_id", "name", "rating", "review_count", "address",
            "lat", "lon", "types", "opening_hours", "website", "phone", "reviews"]:
    if col not in valid_restaurant_list.columns:
        valid_restaurant_list[col] = None
valid_restaurant_list

,full_name,place_id,name,rating,review_count,address,lat,lon,types,opening_hours,website,phone,reviews
0,D.J. REYNOLDS 351 WEST 57 STREET,None,None,None,None,None,None,None,None,None,None,None,None
1,1 EAST 66TH STREET KITCHEN 1 EAST 66 STREET,None,None,None,None,None,None,None,None,None,None,None,None
2,P & S DELI GROCERY 730 COLUMBUS AVENUE,None,None,None,None,None,None,None,None,None,None,None,None
3,ANGELIKA FILM CENTER 18 WEST HOUSTON STREET,None,None,None,None,None,None,None,None,None,None,None,None
4,CAFE METRO 625 8 AVENUE,None,None,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10533,TYPSOON CAFE 947 COLUMBUS AVENUE,None,None,None,None,None,None,None,None,None,None,None,None
10534,DEUX LUXE 384 BROOME STREET,None,None,None,None,None,None,None,None,None,None,None,None
10535,PASTA EATER N GO 744 9 AVENUE,None,None,None,None,None,None,None,None,None,None,None,None
10536,KOL NOODLE 108 WEST 14 STREET,None,None,None,None,None,None,None,None,None,None,None,None


## Step 1: Fetching place info for the top 192 records
This step was halted due to unclear pricing details. To prevent unexpected charges, this method has been suspended for now.

In [ ]:
# # The code below was previously executed, and the resulting data has already been saved to CSV files.
# # To prevent unnecessary re-execution and preserve results, the code has been converted to comments.
# for i, row in valid_restaurant_list.iloc[:192].iterrows():
#     if i % 50 == 0:
#         valid_restaurant_list.to_csv("google_api_restaurant_metadata.csv", index=False)
#         print(f"Checkpoint saved at row {i}")
    
#     if pd.notna(row['place_id']):
#         print(f"Data exists for {row['full_name']}")
#         continue

#     # Step 1: Look up place_id
#     search_url = "https://maps.googleapis.com/maps/api/place/findplacefromtext/json"
#     search_params = {
#         "input": row['full_name'],
#         "inputtype": "textquery",
#         "fields": "place_id",
#         "key": api_key
#     }

#     try:
#         sres = requests.get(search_url, params=search_params, timeout=5).json()
#         candidates = sres.get("candidates", [])

#         if not candidates:
#             valid_restaurant_list.at[i, "place_id"] = False  # 標記查不到
#             continue

#         place_id = candidates[0]['place_id']
#         valid_restaurant_list.at[i, "place_id"] = place_id

#         # Step 2: Fetch detailed info
#         details_url = "https://maps.googleapis.com/maps/api/place/details/json"
#         details_params = {
#             "place_id": place_id,
#             "fields": {
#                         "name,rating,user_ratings_total,formatted_address,types,geometry,"
#                         "opening_hours,formatted_phone_number,website,reviews,"
#                         "curbside_pickup,delivery,dine_in,editorial_summary,price_level,"
#                         "reservable,serves_beer,serves_breakfast,serves_brunch,serves_dinner,"
#                         "serves_lunch,serves_vegetarian_food,serves_wine,takeout"
#                 },
#             "key": api_key
#         }

#         dres = requests.get(details_url, params=details_params, timeout=5).json()
#         result = dres.get("result", {})

#         valid_restaurant_list.at[i, "name"] = result.get("name")
#         valid_restaurant_list.at[i, "rating"] = result.get("rating")
#         valid_restaurant_list.at[i, "review_count"] = result.get("user_ratings_total")
#         valid_restaurant_list.at[i, "address"] = result.get("formatted_address")
#         valid_restaurant_list.at[i, "lat"] = result.get("geometry", {}).get("location", {}).get("lat")
#         valid_restaurant_list.at[i, "lon"] = result.get("geometry", {}).get("location", {}).get("lng")
#         valid_restaurant_list.at[i, "types"] = ", ".join(result.get("types", []))
#         valid_restaurant_list.at[i, "opening_hours"] = "; ".join(
#             result.get("opening_hours", {}).get("weekday_text", []))
#         valid_restaurant_list.at[i, "website"] = result.get("website")
#         valid_restaurant_list.at[i, "phone"] = result.get("formatted_phone_number")
#         valid_restaurant_list.at[i, "reviews"] = result.get("reviews") or []
#         print(i, place_id, result.get("name"), result.get("rating"), result.get("user_ratings_total"))
#         time.sleep(0.2)

#     except Exception as e:
#         print(f"Error on row {i}: {e}")
#         valid_restaurant_list.at[i, "place_id"] = False

# valid_restaurant_list.to_csv("google_api_restaurant_metadata.csv", index=False)
# print(f"Checkpoint saved at row {i}")

## Step 2: Fetch Popular Time Data for Clustered Restaurants
To retrieve popular time data, the [populartimes](https://github.com/m-wrzr/populartimes?tab=readme-ov-file)] package is used. Given the API request limitations, we prioritize restaurants located in dense clusters (e.g., city center). A total of 672 restaurants were processed in this step.

In [33]:
restaurant_df.columns

Index(['Unnamed: 0', 'CAMIS', 'DBA', 'BUILDING', 'STREET', 'ZIPCODE', 'PHONE',
       'CUISINE DESCRIPTION', 'INSPECTION DATE', 'ACTION', 'VIOLATION CODE',
       'VIOLATION DESCRIPTION', 'CRITICAL FLAG', 'SCORE', 'GRADE',
       'GRADE DATE', 'RECORD DATE', 'INSPECTION TYPE', 'Latitude', 'Longitude',
       'Community Board', 'Council District', 'Census Tract', 'BIN', 'BBL',
       'NTA', 'inspection_times', 'violation_times', 'full_name'],
      dtype='object')

In [55]:
valid_restaurant_df = restaurant_df[['full_name', 'Latitude', 'Longitude']].drop_duplicates()
valid_restaurant_df['rough_lat'] = np.round(valid_restaurant_df['Latitude'], 2)
valid_restaurant_df['rough_lng'] = np.round(valid_restaurant_df['Longitude'], 2)
# popular area: 40.75, -73.99
valid_restaurant_df.groupby(['rough_lat', 'rough_lng'])['full_name'].count().sort_values(ascending=False)

rough_lat  rough_lng
40.75      -73.99       672
40.76      -73.98       591
           -73.99       544
40.73      -73.99       516
40.75      -73.98       505
                       ... 
40.70      -74.02         1
40.84      -73.93         1
40.75      -73.96         1
40.78      -73.94         1
40.77      -74.00         1
Name: full_name, Length: 76, dtype: int64

In [48]:
# # The code below was previously executed, and the resulting data has already been saved to CSV files.
# # To prevent unnecessary re-execution and preserve results, the code has been converted to comments.

# valid_restaurant_list = pd.read_csv('google_api_restaurant_metadata.csv')
# valid_restaurant_list = valid_restaurant_list.merge(valid_restaurant_df, on='full_name')
# valid_restaurant_list['clustered'] = np.where((valid_restaurant_list['rough_lat'] == 40.75) & (valid_restaurant_list['rough_lng'] == -73.99), True, False)

rough_lat  rough_lng
40.75      -73.99       672
40.76      -73.98       591
           -73.99       544
40.73      -73.99       516
40.75      -73.98       505
                       ... 
40.70      -74.02         1
40.84      -73.93         1
40.75      -73.96         1
40.78      -73.94         1
40.77      -74.00         1
Name: full_name, Length: 76, dtype: int64

In [50]:
# # The code below was previously executed, and the resulting data has already been saved to CSV files.
# # To prevent unnecessary re-execution and preserve results, the code has been converted to comments.

# Fetch place_id for the clustered restaurant
# for i, row in valid_restaurant_list[valid_restaurant_list['clustered'] == True].iterrows():
#     if pd.notna(row['place_id']):
#         print(f"{row['full_name']} place_id exists: {row['place_id']}")
#         continue

#     place_id = fetch_google_place_id(api_key=api_key,
#                                      place_name=row['full_name'])

#     valid_restaurant_list.at[i, 'place_id'] = place_id
#     print(i, row['full_name'], place_id)
# valid_restaurant_list.to_csv('google_api_restaurant_metadata.csv', index=False)

ANDREW'S NYC DINER 463 7 AVENUE place_id exists: ChIJRdi1GaxZwokR3vmYZ_GDcSM
FIT CAFETERIA (BUILDING A ) 227 WEST   27 STREET place_id exists: False
ARNO RISTORANTE 141 WEST   38 STREET place_id exists: ChIJV0bfnqtZwokRo0b29CRqsS8
HAN BAT RESTAURANT 53 WEST   35 STREET place_id exists: ChIJR751tVNu4oYRGQbCiV0obhs
LAZZARA'S PIZZA CAFE & RESTAURANT 221 WEST   38 STREET place_id exists: ChIJASwLqU9ZwokRAoIgy6k2RgY
245 RICK'S CABARET 50 WEST   33 STREET ChIJ2VhDH6lZwokR_acLQfV63Tc
290 THE TRIPLE CROWN 330 7 AVENUE ChIJDUiwlK9ZwokRgEzA-oCI9a4
348 ZARO'S BREAD BASKET 1 PENNSYLVANIA STATION (AMTRACK) ChIJ30krXK5ZwokRN_iwisXhHxI
392 MUSTANG HARRY'S 352 SEVENTH AVENUE ChIJ6bXru69ZwokRQQ77x79D89M
402 DON PEPI PIZZA 000 PENN STATION ChIJiag-Wq5ZwokRF7-3vlvEWJQ
417 TICK TOCK DINER/ BUTCHER AND THE BANKER 481 8 AVENUE ChIJKUvC6K1ZwokR-6gilCP_YRg
418 TRATTORIA BIANCA 481 8 AVENUE ChIJGwGp6q1ZwokRZAEGF2dtd9c
435 WAKAMBA 543 8 AVENUE ChIJl60eBK1ZwokReIc25YcPsTA
443 STARBUCKS 370 7 AVENUE ChIJpU_RVK5Zw

In [105]:
# # The code below was previously executed.
# # To prevent unnecessary re-execution and preserve results, the code has been converted to comments.

# valid_restaurant_list = pd.read_csv('google_api_restaurant_metadata.csv')
# valid_restaurant_list['time_spent_min'] = None
# valid_restaurant_list['time_spent_max'] = None
# valid_restaurant_list['popular_data'] = False

In [119]:
updated_popularity_rows = []
restaurant_metadata = pd.read_csv('google_api_restaurant_metadata_step2.csv')
restaurant_popular_time = pd.read_csv('google_api_restaurant_popular_time_step2.csv')

cond1 = (restaurant_metadata['clustered'] == True)
cond2 = (restaurant_metadata['place_id'].str.len() > 10)

# Todo: Remove .head(10) and fetch the remaining records 
for i, row in restaurant_metadata[cond1 & cond2].head(10).iterrows():
    place_id = row['place_id']
    
    if row.get('popular_data') is True:
        print(f"{row['place_id']} has been processed")
        continue

    try:
        response = populartimes.get_id(api_key=api_key, place_id=place_id)
        place_info_df, popularity_df = gen_place_info_popularity_df(response)

        for col in place_info_df.columns:
            restaurant_metadata.at[i, col] = place_info_df.at[0, col]
        
        restaurant_metadata.at[i, 'popular_data'] = True

        updated_popularity_rows.append(popularity_df)
        print(f"{i}, {row['place_id']} successfully fetched!")

    except Exception as e:
        print(f"❌ Failed at row {i} (place_id={place_id}): {e}")
        continue

restaurant_metadata.to_csv('google_api_restaurant_metadata_step2.csv', index=False)
final_popularity_df = pd.concat(updated_popularity_rows, ignore_index=True)
restaurant_popular_time = pd.concat([restaurant_popular_time, final_popularity_df], ignore_index=True)
restaurant_popular_time.to_csv('google_api_restaurant_popular_time_step2.csv', index=False)

ChIJRdi1GaxZwokR3vmYZ_GDcSM has been processed
ChIJV0bfnqtZwokRo0b29CRqsS8 has been processed
ChIJR751tVNu4oYRGQbCiV0obhs has been processed
175, ChIJASwLqU9ZwokRAoIgy6k2RgY successfully fetched!
245, ChIJ2VhDH6lZwokR_acLQfV63Tc successfully fetched!
290, ChIJDUiwlK9ZwokRgEzA-oCI9a4 successfully fetched!
348, ChIJ30krXK5ZwokRN_iwisXhHxI successfully fetched!
392, ChIJ6bXru69ZwokRQQ77x79D89M successfully fetched!
402, ChIJiag-Wq5ZwokRF7-3vlvEWJQ successfully fetched!
417, ChIJKUvC6K1ZwokR-6gilCP_YRg successfully fetched!


## View current results

In [121]:
restaurant_metadata = pd.read_csv('google_api_restaurant_metadata_step2.csv')
restaurant_popular_time = pd.read_csv('google_api_restaurant_popular_time_step2.csv')

In [122]:
restaurant_metadata[restaurant_metadata['popular_data'] == True]['place_id'].unique()

array(['ChIJRdi1GaxZwokR3vmYZ_GDcSM', 'ChIJV0bfnqtZwokRo0b29CRqsS8',
       'ChIJR751tVNu4oYRGQbCiV0obhs', 'ChIJASwLqU9ZwokRAoIgy6k2RgY',
       'ChIJ2VhDH6lZwokR_acLQfV63Tc', 'ChIJDUiwlK9ZwokRgEzA-oCI9a4',
       'ChIJ30krXK5ZwokRN_iwisXhHxI', 'ChIJ6bXru69ZwokRQQ77x79D89M',
       'ChIJiag-Wq5ZwokRF7-3vlvEWJQ', 'ChIJKUvC6K1ZwokR-6gilCP_YRg'],
      dtype=object)

In [123]:
restaurant_metadata[restaurant_metadata['popular_data'] == True].head()

,full_name,place_id,name,rating,review_count,address,lat,lon,types,opening_hours,...,phone,reviews,Latitude,Longitude,rough_lat,rough_lng,clustered,time_spent_min,time_spent_max,popular_data
107,ANDREW'S NYC DINER 463 7 AVENUE,ChIJRdi1GaxZwokR3vmYZ_GDcSM,Andrews NYC Diner,4.4,8048.0,"463 7th Ave, New York, NY 10018, USA",40.7517,-73.9898,"restaurant,food,point_of_interest,establishment",Monday: 6:00 AM – 10:00 PM; Tuesday: 6:00 AM –...,...,+1 212-695-1962,"[{'author_name': 'Dibo Goodman G7', 'author_ur...",40.7517,-73.9901,40.75,-73.99,True,60.0,60.0,True
145,ARNO RISTORANTE 141 WEST 38 STREET,ChIJV0bfnqtZwokRo0b29CRqsS8,Arno,4.4,865.0,"Arno Ristorante, 141 W 38th St, New York, NY 1...",40.7534,-73.9880,"bar,restaurant,food,point_of_interest,establis...",Monday: 11:45 AM – 10:00 PM; Tuesday: 11:45 AM...,...,+1 212-944-7420,"[{'author_name': 'Rachel Shalit', 'author_url'...",40.7531,-73.9877,40.75,-73.99,True,180.0,180.0,True
172,HAN BAT RESTAURANT 53 WEST 35 STREET,ChIJR751tVNu4oYRGQbCiV0obhs,Han Bat,4.0,960.0,"53 W 35th St, New York, NY 10001, USA",40.7502,-73.9863,"restaurant,food,point_of_interest,establishment",Monday: 11:30 AM – 10:30 PM; Tuesday: 11:30 AM...,...,+1 212-629-5588,"[{'author_name': 'Hyekyung Kim', 'author_url':...",40.7499,-73.9859,40.75,-73.99,True,NaN,NaN,True
175,LAZZARA'S PIZZA CAFE & RESTAURANT 221 WEST 3...,ChIJASwLqU9ZwokRAoIgy6k2RgY,Lazzara's Pizza Cafe,4.3,965.0,"221 W 38th St #2, New York, NY 10018, USA",40.7540,-73.9895,"restaurant,point_of_interest,food,establishment",Monday: 11:30 AM – 9:00 PM; Tuesday: 11:30 AM ...,...,+1 212-944-7792,"[{'author_name': 'Cristina Padilla', 'author_u...",40.7539,-73.9896,40.75,-73.99,True,90.0,90.0,True
245,RICK'S CABARET 50 WEST 33 STREET,ChIJ2VhDH6lZwokR_acLQfV63Tc,Rick's Cabaret New York,4.4,665.0,"50 W 33rd St, New York, NY 10001, USA",40.7486,-73.9874,"restaurant,food,point_of_interest,establishment",NaN,...,+1 212-372-0850,NaN,40.7486,-73.9872,40.75,-73.99,True,NaN,NaN,True


In [124]:
restaurant_popular_time['place_id'].unique()

array(['ChIJ03By9lhYwokRfGgIPEcxNlM', 'ChIJRdi1GaxZwokR3vmYZ_GDcSM',
       'ChIJV0bfnqtZwokRo0b29CRqsS8', 'ChIJR751tVNu4oYRGQbCiV0obhs',
       'ChIJASwLqU9ZwokRAoIgy6k2RgY', 'ChIJ30krXK5ZwokRN_iwisXhHxI',
       'ChIJ6bXru69ZwokRQQ77x79D89M'], dtype=object)